In [ ]:
import sqlite3
import pandas as pd

## 1. Connexion à la base SQLite
### 'example.db' sera créé si il n'existe pas.

In [ ]:
conn = sqlite3.connect("example.db")
cur = conn.cursor()

## 2. Création de la table

In [ ]:
cur.execute("DROP TABLE IF EXISTS actions")  # On supprime la table si elle existe
cur.execute("""
    CREATE TABLE actions (
        date TEXT,        -- Date de la transaction
        trans TEXT,       -- Type de transaction (BUY/SELL)
        action TEXT,      -- Nom de l'action
        quantite REAL,    -- Quantité d'actions
        prix REAL         -- Prix par action
    )
""")


### 3. Insertion de données

In [ ]:
cur.execute("INSERT INTO actions VALUES ('2020-01-05','BUY','EPSILON',100,35.14)")
cur.execute("INSERT INTO actions VALUES ('2020-01-06','SELL','EPSILON',100,42)")

mouvements = [
    ('2020-03-28', 'BUY', 'IBM', 1000, 45.00),
    ('2020-04-05', 'BUY', 'MSFT', 1000, 72.00),
    ('2020-05-05', 'BUY', 'MSFT', 1000, 74.00),
    ('2020-06-05', 'BUY', 'MSFT', 1000, 61.00),
    ('2020-07-05', 'BUY', 'MSFT', 1000, 55.00),
    ('2020-04-06', 'SELL', 'IBM', 500, 53.00),
    ('2020-01-05','BUY','EPSILON',200,38),
    ('2020-02-05','SELL','EPSILON',300,35.14),
    ('2020-03-05','BUY','EPSILON',100,36),
    ('2020-04-05','BUY','EPSILON',500,37),
]
cur.executemany("INSERT INTO actions VALUES (?,?,?,?,?)", mouvements)
conn.commit()  # Toujours commit pour sauvegarder les modifications

### 4. Lecture (SELECT) avec SQLite
#### On peut récupérer les données directement depuis SQLite

In [ ]:
cur.execute("SELECT * FROM actions WHERE action='EPSILON'")
rows = cur.fetchall()  # Récupère toutes les lignes
print("Transactions EPSILON :")
for row in rows:
    print(row)


### 5. Lecture avec pandas

In [ ]:
df = pd.read_sql_query("SELECT * FROM actions", conn)
print("\nAperçu de toutes les transactions :")
print(df.head())

### 6. Analyses simples avec pandas

#### Total d'actions achetées et vendues

In [ ]:
print("\nTotal d'actions par type de transaction :")
print(df.groupby("trans")["quantite"].sum())

#### Dépenses totales par action (pour les achats)


In [ ]:
print("\nTotal dépensé par action (BUY) :")
df_buy = df[df["trans"]=="BUY"].copy()
df_buy["depense"] = df_buy["quantite"] * df_buy["prix"]
print(df_buy.groupby("action")["depense"].sum())

#### Prix moyen de chaque action achetée

In [ ]:
print("\nPrix moyen par action achetée :")
print(df_buy.groupby("action")["prix"].mean())

#### Transaction la plus chère

In [ ]:
print("\nTransaction la plus chère :")
print(df.loc[df["prix"].idxmax()])

#### Transaction la moins chère

In [ ]:
print("\nTransaction la moins chère :")
print(df.loc[df["prix"].idxmin()])


### 7. Fermeture de la connexion

In [ ]:
conn.close()